In [ ]:
!pip install Craft-xai
!pip install -q timm
!pip install tensorflow
!export HF_HOME=/mnt/abka03/huggingface/hub/

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn

from torchvision import transforms
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from PIL import Image
from qwen_vl_utils import process_vision_info

# cut the model in two


In [3]:
# VLM
import os
import sys
from types import SimpleNamespace
sys.path.append("/mnt/abka03/Projects/xl-vlms/src")

os.environ["HF_HOME"] = "/mnt/abka03/huggingface/hub"

from helpers.logger import log_args, setup_logger
args = SimpleNamespace(**{"local_files_only": False})
save_dir = "/mnt/abka03/Projects/xl-vlms/playground/temp"
text = "\n Describe the imge"
image_path = '/mnt/abka03/mscoco2014/xl-vlm/combined_dataset_coco/train2014/COCO_train2014_000000502827_patch_3.png'
generation_mode = True 
response = ""
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model_name_or_path = "Qwen/Qwen2-VL-7B-Instruct"
processor_name = None
logger = setup_logger(log_file=os.path.join(save_dir, f"logs.log"))



In [ ]:
from transformers import AutoModelForVision2Seq, AutoProcessor
#model = model_class.get_model()
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model_name_or_path = "Qwen/Qwen2-VL-7B-Instruct"
processor = AutoProcessor.from_pretrained(model_name_or_path)
model = AutoModelForVision2Seq.from_pretrained(model_name_or_path, torch_dtype=torch.float16,
            low_cpu_mem_usage=True)
model = model.to(device)

In [5]:
from torch.utils.data import Dataset, DataLoader
import os
class ImageDataset(Dataset):
    def __init__(self, image_dir, prompt="Describe this image in detail:"):
        self.image_paths = [os.path.join(image_dir, img) for img in os.listdir(image_dir) if img.endswith(('.jpg', '.png', '.jpeg'))]
        self.default_prompt = prompt
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        message =  [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": self.image_paths[idx]},
            {"type": "text", "text": self.default_prompt},
        ],
    }
]
        
        return message
def message_collate_fn(batch):
    # This returns a list of messages instead of trying to stack them
    return batch

In [ ]:
image_dir = "/mnt/abka03/xai"



import os
from PIL import Image

# Set the directory containing images
directory = image_dir  # Change this to your directory
new_shape = (224, 224)  # Replace with your desired dimensions (width, height)

# Process each file in the directory
for filename in os.listdir(directory):
    filepath = os.path.join(directory, filename)
    
    # Check if it is an image
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.gif')):
        try:
            # Load the image
            with Image.open(filepath) as img:
                # Resize the image
                resized_img = img.resize(new_shape)
                
                # Convert to RGB (JPEG doesn't support transparency)
                resized_img = resized_img.convert("RGB")
                
                # Create a new filename with .jpg extension
                new_filename = os.path.splitext(filename)[0] + ".jpg"
                new_filepath = os.path.join(directory, new_filename)
                
                # Save the resized image as JPG
                resized_img.save(new_filepath, "JPEG", quality=90)
            
            # Delete the original image
            if filepath != new_filepath:  # Prevent deleting already converted JPGs
                os.remove(filepath)
        
        except Exception as e:
            print(f"Error processing {filename}: {e}")

print("Processing complete.")


# Custom collate function


In [7]:
prompt = "What objects can you see in this image?"  # Customize the prompt as needed
dataset = ImageDataset(image_dir, prompt)


dataloader = DataLoader(
    dataset, 
    batch_size=2,
    shuffle=True,
    collate_fn=message_collate_fn
)



In [ ]:

all_results = []
all_paths = []
count = 5
print(len(dataloader))
with torch.no_grad():
    for i, batch in enumerate(dataloader):
        # Move batch to device
        if i > 5: break
        texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch]
        image_inputs, video_inputs = process_vision_info(batch)

        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda") 



        # Generate text based on inputs
        generated_ids = model.generate(**inputs, max_new_tokens=25, do_sample=False,)
        generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_texts = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        print(output_texts)   

In [ ]:
#model_part1, model_part2 = split_model_at_layer(model, 'model.norm')

# Forward example# Second part
#['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw']
batch = next(iter(dataloader))
texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch]
image_inputs, video_inputs = process_vision_info(batch)

inputs = processor(
    text=texts,
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda") 

print("input_ids: ", inputs['input_ids'].shape)
print("attention_mask: ", inputs['attention_mask'].shape)
print("pixel_values: ", inputs['pixel_values'].shape)
print('image_grid_thw: ' , inputs['image_grid_thw'].shape)


out = model.generate(
                **inputs,max_new_tokens=100,
            do_sample=False,
            output_scores=False,
            return_dict_in_generate=False)




input_len = (
            inputs["input_ids"].shape[1]
            if inputs["input_ids"].ndim > 1
            else inputs["input_ids"].shape[0]
        )

predicted_tokens =  out[:, input_len:]
language_output = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )


print(len(predicted_tokens))
print(language_output)


In [10]:
import torch
class ModelWrapper(torch.nn.Module):
    def __init__(self, model, layer_name):
        super().__init__()
        self.model = model
        self.layer_name = layer_name
        self.embedding = []  # Stores extracted activations
        self.input_keys = ['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw']
        # Register a forward hook to capture activations from the specified layer
        self.max_tokens = 25
        def hook_func(module, inp, out):
            if self.embedding is not None:
                self.embedding.append(out)
        
        # Find the specified layer and attach a hook
        for name, module in self.model.named_modules():
            if name == layer_name:
                module.register_forward_hook(hook_func)
                break
        else:
            raise ValueError(f"Layer {layer_name} not found in model.")
    def remove_number(self, lst, num):
        return [[item for item in sublist if item != num] for sublist in lst]
  
    def get_segment_before_keyword(self, long_strings, short_string):
        sub_strings =  [ ]
        if isinstance(short_string, str):
            short_string = [short_string]
        for strings, sentaces in zip(short_string, long_strings):
            index =  sentaces.find(strings)
            if index != -1:  # If short string is found
                sub_strings.append( sentaces[:index + len(strings)])
            else: sub_strings.append(sentaces)
        return sub_strings 
    def _get_max_new_tokens(self, token_of_interest, original_prediction, processor):
        to_be_prdicted_string = self.get_segment_before_keyword(original_prediction, token_of_interest)
        tokens_all_samples = processor(text=to_be_prdicted_string, return_tensors="pt", padding=True, truncation=False)
        
        #print(tokens_all_samples["input_ids"])
        #
        
        token_list = tokens_all_samples["input_ids"]
       
        # Filter the padding as they are generated for kepiing the tensor row same 
        token_list = self.remove_number(token_list, 151643)
        output_texts = processor.batch_decode(
        token_list, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        tokens_len = [len( token_ids) for token_ids in   token_list ]
    
        return tokens_len
    
    def forward(self, **inputs):
        self.embedding = [] 
        """Runs the model and extracts intermediate embeddings"""
        self.token_of_interest = inputs["token_of_interest"]
        del inputs["token_of_interest"]
        self.processor = inputs["processor"]
        del inputs["processor"]
        filtered_data = {k: v for k, v in inputs.items() if k in self.input_keys}


        tokens_output = self.generate(
            **filtered_data, max_new_tokens=self.max_tokens,
            do_sample=False,
            output_scores=False,
            return_dict_in_generate=False)
      
        #token_logits.argsort(dim=-1, descending=True)#[:,:, 0]
        #top_token_idx = top_token_idx.view(-1)
        #print(top_token_idx)
        tokens_output = tokens_output[0].tolist()
        input_tokens_lens = [len(tokens) for tokens in inputs["input_ids"]]
        tokens_output = [output[t:] for t, output in zip(input_tokens_lens, tokens_output)]
        output = self.processor.batch_decode(tokens_output, skip_special_tokens=True,
           clean_up_tokenization_spaces=False)
        max_tokens_ist = self._get_max_new_tokens(self.token_of_interest, output, self.processor)
        #print(max_tokens_ist)
        max_tokens = max(max_tokens_ist)
        print(max_tokens_ist)



        #emb = torch.stack([self.embedding [v-1][i] for i, v in enumerate(max_tokens_ist)], dim=0)
        #emb = torch.stack([torch.mean(self.embedding [v-1], dim = 0) for i, v in enumerate(max_tokens_ist)], dim=0)
        prefered_embedding = []
        for i, v in enumerate(max_tokens_ist):
            if v == self.max_tokens:
                prefered_embedding.append(torch.mean(self.embedding [v-1], dim = 0))
            elif v == 1:
                print("Exception due to: ", self.embedding [v-1][i].shape)
                prefered_embedding.append(self.embedding [v-1][i][-2:-1])
            else:
                prefered_embedding.append(self.embedding [v-1][i])
        emb = torch.stack(prefered_embedding, dim=0)

        # Get the embiding at an idex and then for each bach get the embedding 
        # Will modify this for mean embedding if the to token does not exist in the output
        if emb.shape[1] > 1:
            emb = self.embedding[:, -1, :].unsqueeze(1)
        return emb  # Returns the extracted activation
    
    def generate(self, **inputs):
        """Runs model.generate() and returns activations"""
        self.embedding = []  # Reset before forward pass
        out = self.model.generate(**inputs)
        return out, self.embedding


def split_model(model, embedding_layer_name, prediction_start_layer_name=None):
    """
    Splits a given model into an embedding extractor and a prediction model
    
    Args:
        model: The original model to split
        embedding_layer_name: Name of layer to extract embeddings from
        prediction_start_layer_name: Name of layer to start prediction from
                                    (if None, defaults to embedding_layer_name)
    """
    if prediction_start_layer_name is None:
        prediction_start_layer_name = embedding_layer_name
    
    full_model = ModelWrapper(model, embedding_layer_name)
    
    class EmbeddingModel(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
        
        def forward(self, **inputs):
            """Returns the intermediate embeddings from the specified layer"""
            return self.model(**inputs)
    
    class PredictionModel(torch.nn.Module):
        def __init__(self, model, start_layer_name):
            super().__init__()
            self.model = model.model
            self.start_layer_name = start_layer_name
            
            # Get all layers from start_layer_name to the end
            self.layers = []
            self.layer_names = []
            self.found_start = False
            
            # Create a dictionary to store all the modules after the starting layer
            for name, module in self.model.named_modules():
                if name == start_layer_name:
                    self.found_start = True
                
                if self.found_start:
                    # Store all subsequent modules
                    self.layer_names.append(name)
                    self.layers.append(module)
            
            if not self.found_start:
                raise ValueError(f"Layer {start_layer_name} not found in model.")
        
        def forward(self, embedding):
            """Runs the model using extracted embeddings"""
            # Start with the embedding
            x = embedding
            lm_head = self.model.lm_head
            # Process through subsequent layers
            # This approach requires knowledge of model architecture
            # Implementation depends on model type (e.g., GPT, T5, etc.)
            
            # For transformer models, we could implement like this:
            # (You'll need to adapt this based on your specific model architecture)
            """
            found_layer = False
            for name, module in self.model.named_modules():
                if found_layer:
                    if hasattr(module, 'forward') and callable(getattr(module, 'forward')):
                        try:
                            x = module(x)
                        except Exception as e:
                            pass  # Skip layers that can't handle the input
                
                if name == self.start_layer_name:
                    found_layer = True
            
            # Final output through lm_head if not already processed
            """
            device = lm_head.weight.device
            dtype = lm_head.weight.dtype
            x = x.to(device=device, dtype=dtype)
            x = lm_head(x)
            return x
    
    embedding_model = EmbeddingModel(full_model)
    prediction_model = PredictionModel(full_model, prediction_start_layer_name)
    
    return embedding_model, prediction_model

In [54]:
def transform_fn(sample, processor, device="cuda"):
    text = processor.apply_chat_template(sample, tokenize=False, add_generation_prompt=True)
    image_input, video_input = process_vision_info(sample)  # Process single sample
    image_paths = [sample_i ["content"][0]["image"] for sample_i in sample] 
    image_name = f" {os.path.split(image_paths[0])[-1].split('_')[0]}"

    inputs = processor(
        text=[text],  # Wrap in list to maintain expected format
        images=[image_input] if image_input is not None else None,
        videos=[video_input] if video_input is not None else None,
        padding=True,
        return_tensors="pt",
    )
    inputs.update({"image_paths": image_paths})
    inputs.update({"token_of_interest": image_name})
    inputs.update({"processor": processor})
    return inputs.to(device)
all_inputs = [transform_fn(inputs, processor) for inputs in dataset]

In [ ]:
print(all_inputs[0])

In [ ]:
# Assuming `model` is already loaded

layer_name = "model.norm"  # Change this to your desired layer
prediction = language_output[0]

inputs = all_inputs[0]
print(inputs['input_ids'].shape)
embedding_model, prediction_model = split_model(model, layer_name)
inputs.update({"token_of_interest": " cat", "processor": processor })
# Get embeddings
print(inputs.keys() )
embedding = embedding_model(**inputs)
print(embedding.shape)
# Generate using embeddings
output = prediction_model(embedding)

token_logits = output.float()
print(output.shape)
top_token_idx = token_logits.argmax(dim=2).squeeze(1)
#token_logits.argsort(dim=-1, descending=True)#[:,:, 0]
#top_token_idx = top_token_idx.view(-1)
#print(top_token_idx)
print(top_token_idx)
predicted_token = processor.decode(top_token_idx, skip_special_tokens=True,
           return_tensors="pt")
print(predicted_token)

In [ ]:
from timm.data.transforms_factory import create_transform
config = resolve_data_config({}, model=model)
transform = create_transform(**config)
rabbit_class = 330 # imagenet class for rabbit
to_pil = transforms.ToPILImage()
# loading some images of rabbits !
images = np.load('assets/rabbit.npz')['arr_0'].astype(np.uint8)
images_preprocessed = torch.stack([transform(to_pil(img)) for img in images], 0)

images_preprocessed.shape

In [58]:
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
from PIL import Image
dtype = torch.float16


images_np = np.load('assets/rabbit.npz')['arr_0'].astype(np.uint8)
images = [to_pil(img).convert("RGB") for img in images_np]

# Step 2: Define the prompt and prepare inputs


In [59]:

"""
CRAFT Module for Tensorflow
"""

from abc import ABC, abstractmethod
from math import ceil
import shutil
from typing import Callable, Optional
from PIL import Image
import torch
import torchvision.utils as vutils
import numpy as np
from sklearn.decomposition import NMF
from sklearn.exceptions import NotFittedError

from craft.sobol.sampler import HaltonSequence
from craft.sobol.estimators import JansenEstimator
from sklearn.decomposition import PCA, DictionaryLearning

def torch_to_numpy(tensor):
  try:
    return tensor.detach().cpu().numpy()
  except:
    return np.array(tensor)


def _combine_batches(one_batch):

    batch_keys = one_batch[0].keys()
    new_batch = {}
    for key in batch_keys:
        
        
        list_of_list = []
        for item in one_batch:
            list_of_list.append(item[key])
            #print(list_of_list)
        if key in ['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw']:
            tensors_value = torch.cat(list_of_list, dim=0)
            new_batch[key] = tensors_value
        elif key in ["token_of_interest", 'image_paths']:
             new_batch[key] = list_of_list
        else:
            new_batch[key] = item[key]
    return new_batch

def _batch_inference(model, dataset, batch_size=128, resize=None, device='cuda'):
  nb_batchs = ceil(len(dataset) / batch_size)
  start_ids = [i*batch_size for i in range(nb_batchs)]

  results = []

  with torch.no_grad():
    for i in start_ids:
      x = dataset[i:i+batch_size]
      #x = x.to(device)
      x = _combine_batches(x)
      #if resize:
      #  x = torch.nn.functional.interpolate(x, size=resize, mode='bilinear', align_corners=False)
      
      results.append(model(**x).cpu())

  results = torch.cat(results)
  return results

def _batch_inference_letent_to_logit(model, dataset, batch_size=128, resize=None, device='cuda'):
  nb_batchs = ceil(len(dataset) / batch_size)
  start_ids = [i*batch_size for i in range(nb_batchs)]

  results = []

  with torch.no_grad():
    for i in start_ids:
      x = torch.tensor(dataset[i:i+batch_size])
      x = x.to(device)

      if resize:
        x = torch.nn.functional.interpolate(x, size=resize, mode='bilinear', align_corners=False)
      x = x.unsqueeze(1)
      results.append(model(x).cpu())

  results = torch.cat(results)
  return results


class BaseConceptExtractor(ABC):
    """
    Base class for concept extraction models.

    Parameters
    ----------
    input_to_latent : Callable
        The first part of the model taking an input and returning
        positive activations, g(.) in the original paper.
    latent_to_logit : Callable
        The second part of the model taking activation and returning
        logits, h(.) in the original paper.
    number_of_concepts : int
        The number of concepts to extract.
    batch_size : int, optional
        The batch size to use during training and prediction. Default is 64.

    """

    def __init__(self, input_to_latent : Callable,
                       latent_to_logit : Optional[Callable] = None,
                       number_of_concepts: int = 20,
                       batch_size: int = 64):

        # sanity checks
        assert(number_of_concepts > 0), "number_of_concepts must be greater than 0"
        assert(batch_size > 0), "batch_size must be greater than 0"
        assert(callable(input_to_latent)), "input_to_latent must be a callable function"

        self.input_to_latent = input_to_latent
        self.latent_to_logit = latent_to_logit
        self.number_of_concepts = number_of_concepts
        self.batch_size = batch_size

    @abstractmethod
    def fit(self, inputs):
        """
        Fit the CAVs to the input data.

        Parameters
        ----------
        inputs : array-like
            The input data to fit the model on.

        Returns
        -------
        tuple
            A tuple containing the input data and the matrices (U, W) that factorize the data.

        """
        raise NotImplementedError

    @abstractmethod
    def transform(self, inputs):
        """
        Transform the input data into a concepts embedding.

        Parameters
        ----------
        inputs : array-like
            The input data to transform.

        Returns
        -------
        array-like
            The transformed embedding of the input data.

        """
        raise NotImplementedError


class Craft(BaseConceptExtractor):
    """
    Class Implementing the CRAFT Concept Extraction Mechanism.

    Parameters
    ----------
    input_to_latent : Callable
        The first part of the model taking an input and returning
        positive activations, g(.) in the original paper.
    latent_to_logit : Callable, optional
        The second part of the model taking activation and returning
        logits, h(.) in the original paper.
    number_of_concepts : int
        The number of concepts to extract.
    batch_size : int, optional
        The batch size to use during training and prediction. Default is 64.
    patch_size : int, optional
        The size of the patches to extract from the input data. Default is 64.
    """

    def __init__(self, input_to_latent: Callable,
                       latent_to_logit: Optional[Callable] = None,
                       number_of_concepts: int = 20,
                       batch_size: int = 64,
                       patch_size: int = 64,
                       device : str = 'cuda'):
        super().__init__(input_to_latent, latent_to_logit, number_of_concepts, batch_size)

        self.patch_size = patch_size
        self.activation_shape = None
        self.device = device



    def fit(self, inputs: np.ndarray):
        """
        Fit the Craft model to the input data.

        Parameters
        ----------
        inputs : np.ndarray
            Preprocessed Iinput data of shape (n_samples, channels, height, width).
            (x1, x2, ..., xn) in the paper.

        Returns
        -------
        (X, U, W)
            A tuple containing the crops (X in the paper),
            the concepts values (U) and the concepts basis (W).
        """
        #assert len(inputs.shape) == 4, "Input data must be of shape (n_samples, channels, height, width)."
        #assert inputs.shape[2] == inputs.shape[3], "Input data must be square."
        image_size = (224, 224)

        transform = transforms.Compose([
        transforms.Resize(image_size),  # Resize first
            transforms.ToTensor()  # Convert to tensor
        ])
         
        #image_size = inputs[0]["image_path"]
        # extract patches from the input data, keep patches on cpu
        strides = int(self.patch_size * 0.80)
        all_images = torch.stack([
            transform(Image.open(samp.image_paths[0]).convert("RGB")) for samp in inputs
        ])
        patches = torch.nn.functional.unfold(all_images , kernel_size=self.patch_size, stride=strides)
        patches = patches.transpose(1, 2).contiguous().view(-1, 3, self.patch_size, self.patch_size)
        

        # Directory to save images
        save_dir = "/mnt/abka03/temp"
        # Delete the directory if it exists
        if os.path.exists(save_dir):
            shutil.rmtree(save_dir)

        os.makedirs(save_dir, exist_ok=True)

        # Save each image
        for i in range(patches.shape[0]):
            vutils.save_image(patches[i], os.path.join(save_dir, f"image_{i}.png"))
        patch_prompt = "What is in the image?"
        dataset = ImageDataset(save_dir, patch_prompt)
        all_patch_inputs = [transform_fn(inputs, processor) for inputs in dataset]

        # encode the patches and obtain the activations
        activations = _batch_inference(self.input_to_latent, all_patch_inputs , self.batch_size, image_size, 
                                       device=self.device)

        #assert torch.min(activations) >= 0.0, "Activations must be positive."

        # if the activations have shape (n_samples, height, width, n_channels),
        # apply average pooling
       
        #if len(activations.shape) == 4:
        #    activations = torch.mean(activations, dim=(2, 3))
        #print(activations.shape)
        activations = activations.squeeze(1)
        # apply NMF to the activations to obtain matrices U and W
        #reducer = NMF(n_components=self.number_of_concepts)
        reducer = DictionaryLearning(
            n_components=self.number_of_concepts,
            positive_code=True,
            fit_algorithm="cd",
            transform_algorithm="lasso_cd",
            max_iter=40000,
        )
        U = reducer.fit_transform(torch_to_numpy(activations))
        W = reducer.components_.astype(np.float32)

        # store the factorizer and W as attributes of the Craft instance
        self.reducer = reducer
        self.W = np.array(W, dtype=np.float32)

        return patches, U, W

    def check_if_fitted(self):
        """Checks if the factorization model has been fitted to input data.

        Raises
        ------
        NotFittedError
            If the factorization model has not been fitted to input data.
        """

        if not hasattr(self, 'reducer'):
            raise NotFittedError("The factorization model has not been fitted to input data yet.")

    def transform(self, inputs: np.ndarray, activations: Optional[np.ndarray] = None):
        self.check_if_fitted()

        if activations is None:
            activations =_batch_inference(self.input_to_latent, inputs, self.batch_size,
                                           device=self.device)
  
            activations = activations.squeeze(1)
        is_4d = len(activations.shape) == 4

        if is_4d:
            # (N, C, W, H) -> (N * W * H, C)
            activation_size = activations.shape[-1]
            activations = activations.permute(0, 2, 3, 1)
            activations = torch.reshape(activations, (-1, activations.shape[-1]))
        W_dtype = self.reducer.components_.dtype

        U = self.reducer.transform(torch_to_numpy(activations).astype(W_dtype))

        if is_4d:
          # (N * W * H, R) -> (N, W, H, R)
          U = np.reshape(U, (-1, activation_size, activation_size, U.shape[-1]))

        return U

    def estimate_importance(self, inputs, class_id, nb_design=32):
        """
        Estimates the importance of each concept for a given class.

        Parameters
        ----------
        inputs : numpy array or Tensor
            The input data to be transformed.
        class_id : int
            The class id to estimate the importance for.
        nb_design : int, optional
            The number of design to use for the importance estimation. Default is 32.

        Returns
        -------
        importances : list
            The Sobol total index (importance score) for each concept.

        """
        self.check_if_fitted()

        U = self.transform(inputs)

        masks = HaltonSequence()(self.number_of_concepts, nb_design=nb_design).astype(np.float32)
        estimator = JansenEstimator()

        importances = []

        if len(U.shape) == 2:
            # apply the original method of the paper

            for u in U:
                u_perturbated = u[None, :] * masks
                a_perturbated = u_perturbated @ self.W
                y_pred = _batch_inference_letent_to_logit(self.latent_to_logit, a_perturbated, self.batch_size,
                                          device=self.device)
                
                
                y_pred = y_pred.squeeze(1) 
   
                y_pred = y_pred[:, class_id]

                stis = estimator(torch_to_numpy(masks),
                                 torch_to_numpy(y_pred),
                                 nb_design)

                importances.append(stis)

        elif len(U.shape) == 4:
            # apply a re-parameterization trick and use mask on all localization for a given
            # concept id to estimate sobol indices
            for u in U:
                u_perturbated = u[None, :] * masks[:, None, None, :]
                a_perturbated = np.reshape(u_perturbated,(-1, u.shape[-1])) @ self.W
                a_perturbated = np.reshape(a_perturbated, (len(masks), U.shape[1], U.shape[2], -1))
                a_perturbated = np.moveaxis(a_perturbated, -1, 1)

                y_pred = _batch_inference(self.latent_to_logit, a_perturbated, self.batch_size,
                                          device=self.device)
                y_pred = y_pred[:, class_id]

                stis = estimator(torch_to_numpy(masks),
                                 torch_to_numpy(y_pred),
                                 nb_design)

                importances.append(stis)

        return np.mean(importances, 0)


In [ ]:
#from lvm_craft import Craft, torch_to_numpy

craft = Craft(input_to_latent = embedding_model,
              latent_to_logit = prediction_model,
              number_of_concepts = 5,
              patch_size = 100,
              batch_size = 6,
              device = device)

# now we can start fit the concept using our rabbit images
# CRAFT will (1) create the patches, (2) find the concept
# and (3) return the crops (crops), the embedding of the crops (crops_u), and the concept bank (w)
crops, crops_u, w = craft.fit(all_inputs)
crops = np.moveaxis(torch_to_numpy(crops), 1, -1)

crops.shape, crops_u.shape, w.shape

In [ ]:
importances = craft.estimate_importance(all_inputs, class_id=8251) # 330 is the rabbit class id in imagenet



In [ ]:
images_u = craft.transform(all_inputs[-3: -1])
images_u.shape

In [ ]:
# We are done, let's inspect the results !
# first, lets see which concepts matter
import matplotlib.pyplot as plt

plt.bar(range(len(importances)), importances)
plt.xticks(range(len(importances)))
plt.title("Concept Importance")

most_important_concepts = np.argsort(importances)[::-1][:10]

for c_id in most_important_concepts:
  print("Concept", c_id, " has an importance value of ", importances[c_id])

In [ ]:
# Ok nice, let inspect those concepts by showing the 10 best crops for
# each concepts
from math import ceil
nb_crops = 5

def show(img, **kwargs):
  img = np.array(img)
  if img.shape[0] == 3:
    img = img.transpose(1, 2, 0)

  img -= img.min();img /= img.max()
  plt.imshow(img, **kwargs); plt.axis('off')

for c_id in most_important_concepts:

  best_crops_ids = np.argsort(crops_u[:, c_id])[::-1][:nb_crops]
  best_crops = crops[best_crops_ids]

  print("Concept", c_id, " has an importance value of ", importances[c_id])
  for i in range(nb_crops):
    plt.subplot(ceil(nb_crops/5), 5, i+1)
    show(best_crops[i])
  plt.show()
  print('\n\n')

In [41]:

from matplotlib.colors import ListedColormap
import matplotlib
import colorsys

def get_alpha_cmap(cmap):
  if isinstance(cmap, str):
    cmap = plt.get_cmap(cmap)
  else:
    c = np.array((cmap[0]/255.0, cmap[1]/255.0, cmap[2]/255.0))

    cmax = colorsys.rgb_to_hls(*c)
    cmax = np.array(cmax)
    cmax[-1] = 1.0

    cmax = np.clip(np.array(colorsys.hls_to_rgb(*cmax)), 0, 1)
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", [c,cmax])

  alpha_cmap = cmap(np.arange(256))
  alpha_cmap[:,-1] = np.linspace(0, 0.85, 256)
  alpha_cmap = ListedColormap(alpha_cmap)

  return alpha_cmap

In [ ]:
cmaps = [
  get_alpha_cmap((54, 197, 240)),
  get_alpha_cmap((210, 40, 95)),
  get_alpha_cmap((236, 178, 46)),
  get_alpha_cmap((15, 157, 88)),
  get_alpha_cmap((84, 25, 85))
]

def plot_legend():
  for i, c_id in enumerate(most_important_concepts):
    cmap = cmaps[i]
    plt.subplot(1, len(most_important_concepts), i+1)

    best_crops_id = np.argsort(crops_u[:, c_id])[::-1][0]
    best_crop = crops[best_crops_id]

    p = 5
    mask = np.zeros(best_crop.shape[:-1])
    mask[:p, :] = 1.0
    mask[:, :p] = 1.0
    mask[-p:, :] = 1.0
    mask[:, -p:] = 1.0

    show(best_crop)
    show(mask, cmap=cmap)
    plt.title(f"{c_id}", color=cmap(1.0))

  plt.show()

plot_legend()

In [ ]:
def concept_attribution_maps(id, percentile=90):
  img = all_inputs[0:30][id]
  u = images_u[id]
  print(u)
  show(img)

  for i, c_id in enumerate(most_important_concepts):

    cmap = cmaps[i]
    heatmap = u[:, :, c_id]

    # only show concept if excess N-th percentile
    sigma = np.percentile(images_u[:,:,:,c_id].flatten(), percentile)
    heatmap = heatmap * np.array(heatmap > sigma, np.float32)

    heatmap = cv2.resize(heatmap[:, :, None], (224, 224))
    show(heatmap, cmap=cmap, alpha=0.7)

  plt.show()

plot_legend()
concept_attribution_maps(0)
plt.show()
concept_attribution_maps(1)
plt.show()
concept_attribution_maps(2)
plt.show()